# Hybrid Search Benchmark Lab

## BM25 vs Dense Retrieval vs Hybrid (RRF) -- Head-to-Head on a QA Dataset

**Project 21** -- Advance RAG Learning Series

| Property | Value |
|----------|-------|
| Task | Information retrieval benchmark |
| Methods | BM25, dense embedding, Reciprocal Rank Fusion (hybrid) |
| Metrics | Precision@k, Recall@k, MRR, Latency |
| Embedding model | all-MiniLM-L6-v2 (sentence-transformers) |
| Corpus | 20 embedded QA documents (tech domain) |
| Queries | 12 evaluation queries with ground-truth relevance labels |


## Project Overview

Every RAG system starts with **retrieval**. The quality of retrieved context
determines everything downstream -- if the retriever misses the right document,
the generator cannot produce a good answer.

Three common approaches:

| Method | Mechanism | Strength |
|--------|-----------|----------|
| **BM25** | TF-IDF-style term matching | Exact keyword match, fast |
| **Dense** | Embedding vector similarity | Semantic meaning, paraphrase-robust |
| **Hybrid** | Combines BM25 + Dense via rank fusion | Best of both worlds |

This notebook benchmarks all three on the **same corpus and queries** to
answer: *When does each method win?*


## Learning Goals

1. Understand how BM25 keyword search works (TF, IDF, length normalisation)
2. Build a dense retriever using sentence-transformers
3. Implement Reciprocal Rank Fusion (RRF) for hybrid search
4. Evaluate retrieval with Precision@k, Recall@k, and MRR
5. Compare retrieval methods across different query types
6. Identify when each method excels and when it fails


## Problem Statement

Given a corpus of technical documents and a set of natural-language queries,
retrieve the most relevant documents for each query.

Each query has **ground-truth relevant document IDs** so we can measure
precision, recall, and ranking quality -- not just eyeball results.

Three query types test different retrieval challenges:
- **Exact match** -- query terms appear verbatim in docs
- **Keyword overlap** -- partial term overlap, some reformulation
- **Semantic** -- meaning match with different vocabulary


## Why This Project Matters

- **RAG performance is retrieval-limited.** A generator's output is only
  as good as the context it receives.
- **No single method dominates.** BM25 wins on exact matches; dense wins
  on semantic paraphrases; hybrid usually wins overall.
- **Cost vs quality tradeoff.** BM25 is CPU-only and sub-millisecond.
  Dense retrieval requires GPU embedding and vector search infrastructure.
- **Every RAG engineer needs this intuition.** Choosing the right retrieval
  method for your data is one of the highest-leverage design decisions.


## Key Concept: Reciprocal Rank Fusion (RRF)

RRF combines two ranked lists by scoring each document:

$$\text{score}(d) = \sum_{i} \frac{1}{k + \text{rank}_i(d)}$$

where $k$ is a smoothing constant (typically 60).

Documents ranked high by **either** retriever get a high fused score.
Documents ranked high by **both** retrievers get the highest.

RRF advantages:
- No score normalisation needed (works on ranks only)
- No training or tuning required
- Simple to implement and reason about


## Environment Setup

In [ ]:
import subprocess, sys, warnings

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["rank-bm25", "sentence-transformers"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

warnings.filterwarnings("ignore")
print("Environment ready.")


## Imports

In [ ]:
import time, random, json
from collections import defaultdict
from typing import List, Dict, Tuple
from dataclasses import dataclass

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports loaded.")


## Configuration

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # fast, good quality for benchmarking
K = 3                                   # evaluate at top-3
RRF_K = 60                              # RRF smoothing constant

print(f"Config: model={EMBEDDING_MODEL}, K={K}, RRF_k={RRF_K}")


## Benchmark Corpus

We use a hand-crafted 20-document tech knowledge base.
Each document has a unique ID so we can track relevance.


In [ ]:
corpus = [
    {"id": "D01", "text": "Python is a high-level programming language known for its readability. It supports multiple paradigms including object-oriented, functional, and procedural programming. Created by Guido van Rossum in 1991."},
    {"id": "D02", "text": "JavaScript is a scripting language used for web development. It runs in browsers and on servers via Node.js. It supports event-driven, functional, and prototype-based programming."},
    {"id": "D03", "text": "Machine learning is a subset of artificial intelligence where models learn from data. Common algorithms include decision trees, support vector machines, and neural networks."},
    {"id": "D04", "text": "Deep learning uses multi-layer neural networks to learn complex patterns. Popular frameworks include TensorFlow, PyTorch, and JAX. It excels in image recognition, NLP, and speech processing."},
    {"id": "D05", "text": "PostgreSQL is an open-source relational database management system. It supports ACID transactions, JSON storage, full-text search, and advanced indexing including GiST and GIN indexes."},
    {"id": "D06", "text": "MongoDB is a NoSQL document database that stores data in BSON format. It offers horizontal scaling through sharding, replica sets for high availability, and a flexible schema design."},
    {"id": "D07", "text": "Docker containers package application code with dependencies into portable units. Containers share the host OS kernel, making them lighter than virtual machines. Docker Compose orchestrates multi-container apps."},
    {"id": "D08", "text": "Kubernetes orchestrates containerized workloads. It handles auto-scaling, service discovery, load balancing, and rolling deployments. Managed K8s is available on AWS EKS, GCP GKE, and Azure AKS."},
    {"id": "D09", "text": "REST APIs use HTTP methods (GET, POST, PUT, DELETE) for CRUD operations. They follow stateless client-server architecture. Common serialization formats include JSON and XML."},
    {"id": "D10", "text": "GraphQL is a query language for APIs that lets clients request exactly the data they need. It uses a typed schema, resolvers, and a single endpoint instead of multiple REST endpoints."},
    {"id": "D11", "text": "Git is a distributed version control system for tracking code changes. It supports branching, merging, and rebasing. Remote hosting is available on GitHub, GitLab, and Bitbucket."},
    {"id": "D12", "text": "Transformers are neural network architectures using self-attention mechanisms. Introduced in Attention Is All You Need paper of 2017. GPT, BERT, and T5 are all transformer-based models."},
    {"id": "D13", "text": "Retrieval-Augmented Generation (RAG) combines a retriever and a generator. The retriever finds relevant documents and the generator produces an answer using those documents as context."},
    {"id": "D14", "text": "Vector databases store high-dimensional embeddings for similarity search. Examples include Pinecone, Weaviate, Qdrant, Milvus, and ChromaDB. They use approximate nearest neighbor (ANN) algorithms."},
    {"id": "D15", "text": "BM25 is a probabilistic ranking function for keyword search. It considers term frequency, inverse document frequency, and document length normalization. It is the baseline for many search systems."},
    {"id": "D16", "text": "FastAPI is a modern Python web framework for building APIs. It uses Pydantic for validation, async support for high concurrency, and automatic OpenAPI documentation generation."},
    {"id": "D17", "text": "Redis is an in-memory data store used as cache, message broker, and database. It supports data structures like strings, hashes, lists, sets, and sorted sets with sub-millisecond latency."},
    {"id": "D18", "text": "Apache Kafka is a distributed event streaming platform. It handles real-time data feeds with high throughput, fault tolerance via replication, and consumer groups for parallel processing."},
    {"id": "D19", "text": "Prompt engineering is the practice of crafting inputs to large language models to guide their output. Techniques include few-shot examples, chain-of-thought reasoning, and role specification."},
    {"id": "D20", "text": "Fine-tuning adapts a pretrained model to a specific task using task-specific data. Methods include full fine-tuning, LoRA (low-rank adaptation), and QLoRA for memory-efficient tuning."},
]

print(f"Corpus: {len(corpus)} documents")
for doc in corpus[:3]:
    print(f"  [{doc['id']}] {doc['text'][:60]}...")
print("  ...")


## Evaluation Queries with Ground-Truth Labels

Each query has:
- **relevant**: list of document IDs that answer the query
- **type**: whether the query needs exact match, keyword overlap, or semantic understanding


In [ ]:
queries = [
    # Exact match -- terms appear verbatim in target docs
    {"query": "What programming language did Guido van Rossum create?",
     "relevant": ["D01"], "type": "exact_match"},
    {"query": "BM25 term frequency inverse document frequency",
     "relevant": ["D15"], "type": "exact_match"},
    {"query": "Attention Is All You Need transformer",
     "relevant": ["D12"], "type": "exact_match"},

    # Keyword overlap -- partial match, some reformulation
    {"query": "database that supports JSON storage",
     "relevant": ["D05", "D06"], "type": "keyword"},
    {"query": "CRUD operations over HTTP",
     "relevant": ["D09"], "type": "keyword"},
    {"query": "in-memory caching data store",
     "relevant": ["D17"], "type": "keyword"},
    {"query": "event streaming high throughput",
     "relevant": ["D18"], "type": "keyword"},

    # Semantic -- meaning match with different vocabulary
    {"query": "how do neural networks learn complex patterns?",
     "relevant": ["D03", "D04", "D12"], "type": "semantic"},
    {"query": "container orchestration and auto-scaling",
     "relevant": ["D07", "D08"], "type": "semantic"},
    {"query": "how does RAG combine search with generation?",
     "relevant": ["D13", "D14"], "type": "semantic"},
    {"query": "tools for managing code versions and branches",
     "relevant": ["D11"], "type": "semantic"},
    {"query": "making pretrained models work on custom data",
     "relevant": ["D20"], "type": "semantic"},
]

print(f"Evaluation queries: {len(queries)}")
type_counts = {}
for q in queries:
    type_counts[q["type"]] = type_counts.get(q["type"], 0) + 1
for qtype, count in type_counts.items():
    print(f"  {qtype}: {count} queries")


## Retriever 1: BM25 (Keyword Search)

BM25 ranks documents by term overlap. It considers:
- **Term frequency (TF)** -- how often the query term appears in the document
- **Inverse document frequency (IDF)** -- rarer terms get higher weight
- **Document length normalisation** -- longer docs are slightly penalised

BM25 is fast (CPU-only), deterministic, and requires no training.


In [ ]:
# Build BM25 index
tokenized_corpus = [doc["text"].lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


def bm25_search(query: str, k: int = 5) -> List[Dict]:
    """BM25 keyword search. Returns ranked docs with scores."""
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [{"id": corpus[i]["id"], "score": float(scores[i]), "rank": rank + 1}
            for rank, i in enumerate(top_indices) if scores[i] > 0]


# Sanity check
test_results = bm25_search("Python programming language", k=3)
print("BM25 retriever built")
print(f"Test query results: {[r['id'] for r in test_results]}")


## Retriever 2: Dense Embedding Search

Dense retrieval encodes queries and documents into the same vector space.
Similarity is measured by cosine distance between embedding vectors.

Dense search excels at **semantic matching** -- queries and documents
can use completely different words but still match by meaning.


In [ ]:
# Build dense index
print(f"Loading embedding model: {EMBEDDING_MODEL}...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

# Pre-encode all documents
doc_texts = [doc["text"] for doc in corpus]
doc_embeddings = encoder.encode(doc_texts, convert_to_numpy=True, show_progress_bar=False)
doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)  # L2 normalise


def dense_search(query: str, k: int = 5) -> List[Dict]:
    """Dense embedding search with cosine similarity."""
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    
    scores = (doc_embeddings @ q_emb.T).flatten()
    top_indices = np.argsort(scores)[::-1][:k]
    return [{"id": corpus[i]["id"], "score": float(scores[i]), "rank": rank + 1}
            for rank, i in enumerate(top_indices)]


# Sanity check
test_results = dense_search("software for managing source code", k=3)
print(f"Dense retriever built (index shape: {doc_embeddings.shape})")
print(f"Test query results: {[r['id'] for r in test_results]}")


## Retriever 3: Hybrid (BM25 + Dense + RRF)

Reciprocal Rank Fusion merges ranked lists from both retrievers.
No score normalisation needed -- it operates on **ranks** only.


In [ ]:
def reciprocal_rank_fusion(
    bm25_results: List[Dict],
    dense_results: List[Dict],
    rrf_k: int = RRF_K,
    top_n: int = 5,
) -> List[Dict]:
    """Fuse two ranked lists using Reciprocal Rank Fusion."""
    fused_scores = defaultdict(float)

    for r in bm25_results:
        fused_scores[r["id"]] += 1.0 / (rrf_k + r["rank"])

    for r in dense_results:
        fused_scores[r["id"]] += 1.0 / (rrf_k + r["rank"])

    sorted_ids = sorted(fused_scores, key=fused_scores.get, reverse=True)[:top_n]
    return [{"id": doc_id, "score": fused_scores[doc_id], "rank": rank + 1}
            for rank, doc_id in enumerate(sorted_ids)]


def hybrid_search(query: str, k: int = 5) -> List[Dict]:
    """Hybrid search = BM25 + Dense + RRF."""
    bm25_results = bm25_search(query, k=k)
    dense_results = dense_search(query, k=k)
    return reciprocal_rank_fusion(bm25_results, dense_results, top_n=k)


# Sanity check
test_results = hybrid_search("container orchestration", k=3)
print(f"Hybrid retriever built")
print(f"Test query results: {[r['id'] for r in test_results]}")


## Evaluation Metrics

| Metric | What it measures |
|--------|------------------|
| **Precision@k** | Fraction of top-k results that are relevant |
| **Recall@k** | Fraction of relevant docs found in top-k |
| **MRR** | Reciprocal rank of the **first** relevant result |

MRR is especially important for QA systems where you need
the best answer at rank 1.


In [ ]:
def precision_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    """Fraction of top-k retrieved docs that are relevant."""
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in top_k if doc_id in relevant_ids)
    return hits / k if k > 0 else 0.0


def recall_at_k(retrieved_ids: List[str], relevant_ids: List[str], k: int) -> float:
    """Fraction of relevant docs that were retrieved in top-k."""
    top_k = retrieved_ids[:k]
    hits = sum(1 for doc_id in relevant_ids if doc_id in top_k)
    return hits / len(relevant_ids) if relevant_ids else 0.0


def mean_reciprocal_rank(retrieved_ids: List[str], relevant_ids: List[str]) -> float:
    """Reciprocal rank of the first relevant result."""
    for i, doc_id in enumerate(retrieved_ids):
        if doc_id in relevant_ids:
            return 1.0 / (i + 1)
    return 0.0


print("Evaluation metrics defined.")


## Run the Benchmark

In [ ]:
results = {"BM25": [], "Dense": [], "Hybrid": []}
latencies = {"BM25": [], "Dense": [], "Hybrid": []}

search_fns = [("BM25", bm25_search), ("Dense", dense_search), ("Hybrid", hybrid_search)]

for q in queries:
    query_text = q["query"]
    relevant = q["relevant"]

    for name, search_fn in search_fns:
        start = time.perf_counter()
        hits = search_fn(query_text, k=K)
        elapsed = time.perf_counter() - start

        retrieved_ids = [h["id"] for h in hits]
        results[name].append({
            "query": query_text,
            "type": q["type"],
            "precision": precision_at_k(retrieved_ids, relevant, K),
            "recall": recall_at_k(retrieved_ids, relevant, K),
            "mrr": mean_reciprocal_rank(retrieved_ids, relevant),
            "retrieved": retrieved_ids,
            "relevant": relevant,
        })
        latencies[name].append(elapsed)

print(f"Benchmark complete: {len(queries)} queries x 3 methods = {len(queries)*3} evaluations")


## Aggregate Results

In [ ]:
print(f"{'Metric':<15} {'BM25':>10} {'Dense':>10} {'Hybrid':>10}")
print("-" * 47)

metric_summary = {}
for metric in ["precision", "recall", "mrr"]:
    row = {}
    for name in ["BM25", "Dense", "Hybrid"]:
        avg = sum(r[metric] for r in results[name]) / len(results[name])
        row[name] = avg
    metric_summary[metric] = row
    best_val = max(row.values())
    vals = ""
    for name in ["BM25", "Dense", "Hybrid"]:
        marker = " *" if row[name] == best_val else ""
        vals += f"{row[name]:>9.3f}{marker} "
    label = f"{metric}@{K}" if metric != "mrr" else "MRR"
    print(f"{label:<15} {vals}")

# Latency
print("-" * 47)
lat_vals = ""
for name in ["BM25", "Dense", "Hybrid"]:
    avg_ms = sum(latencies[name]) / len(latencies[name]) * 1000
    lat_vals += f"{avg_ms:>9.1f}ms "
print(f"{'Latency (avg)':<15} {lat_vals}")


## Per-Query-Type Breakdown

The most interesting insight: **which method wins for which query type?**


In [ ]:
for qtype in ["exact_match", "keyword", "semantic"]:
    print(f"\nQuery Type: {qtype}")
    print(f"{'Method':<10} {'P@'+str(K):>8} {'R@'+str(K):>8} {'MRR':>8}")
    print("-" * 36)

    for name in ["BM25", "Dense", "Hybrid"]:
        type_results = [r for r in results[name] if r["type"] == qtype]
        if not type_results:
            continue
        avg_p = sum(r["precision"] for r in type_results) / len(type_results)
        avg_r = sum(r["recall"] for r in type_results) / len(type_results)
        avg_m = sum(r["mrr"] for r in type_results) / len(type_results)
        print(f"{name:<10} {avg_p:>8.3f} {avg_r:>8.3f} {avg_m:>8.3f}")


## Per-Query Detail

In [ ]:
print(f"Per-Query Recall@{K}:\n")
for i, q in enumerate(queries):
    bm25_r = results["BM25"][i]["recall"]
    dense_r = results["Dense"][i]["recall"]
    hybrid_r = results["Hybrid"][i]["recall"]
    best = max(bm25_r, dense_r, hybrid_r)
    winners = [n for n, v in [("BM25", bm25_r), ("Dense", dense_r), ("Hybrid", hybrid_r)] if v == best]
    print(f"  Q{i+1:2d} [{q['type'][:7]:>7s}] BM25={bm25_r:.2f}  Dense={dense_r:.2f}  "
          f"Hybrid={hybrid_r:.2f}  Winner: {', '.join(winners)}")


## Qualitative Comparison

Let's examine specific queries where the methods disagree.


In [ ]:
print("Qualitative Analysis: Cases Where Methods Disagree\n")
print("=" * 70)

for i, q in enumerate(queries):
    bm25_ids = results["BM25"][i]["retrieved"]
    dense_ids = results["Dense"][i]["retrieved"]
    hybrid_ids = results["Hybrid"][i]["retrieved"]
    relevant = set(q["relevant"])

    # Check if methods disagree on recall
    bm25_r = results["BM25"][i]["recall"]
    dense_r = results["Dense"][i]["recall"]
    if abs(bm25_r - dense_r) > 0.3:   # significant disagreement
        print(f"\nQ{i+1}: {q['query']}")
        print(f"  Type: {q['type']} | Relevant: {q['relevant']}")
        print(f"  BM25 retrieved:   {bm25_ids} (recall={bm25_r:.2f})")
        print(f"  Dense retrieved:  {dense_ids} (recall={dense_r:.2f})")
        print(f"  Hybrid retrieved: {hybrid_ids} (recall={results['Hybrid'][i]['recall']:.2f})")
        if dense_r > bm25_r:
            print(f"  --> Dense wins: semantic understanding found meaning despite vocabulary gap")
        else:
            print(f"  --> BM25 wins: exact term overlap matches better than embedding similarity")

print("\n" + "=" * 70)
print("Summary: Hybrid combines both signals and is rarely the worst performer.")


## Sensitivity Analysis: Varying K

How does performance change as we retrieve more documents?


In [ ]:
print(f"{'k':>3}  {'BM25 R@k':>10}  {'Dense R@k':>10}  {'Hybrid R@k':>10}")
print("-" * 40)

for k_val in [1, 3, 5, 10]:
    row = {}
    for name, search_fn in search_fns:
        recalls = []
        for q in queries:
            hits = search_fn(q["query"], k=k_val)
            retrieved_ids = [h["id"] for h in hits]
            recalls.append(recall_at_k(retrieved_ids, q["relevant"], k_val))
        row[name] = sum(recalls) / len(recalls)
    print(f"{k_val:>3}  {row['BM25']:>10.3f}  {row['Dense']:>10.3f}  {row['Hybrid']:>10.3f}")


## Error Analysis

Examine cases where **all methods fail** and cases where
**only one method succeeds**.


In [ ]:
print("Error Analysis\n")

all_fail = []
only_one_wins = []

for i, q in enumerate(queries):
    scores = {
        "BM25": results["BM25"][i]["recall"],
        "Dense": results["Dense"][i]["recall"],
        "Hybrid": results["Hybrid"][i]["recall"],
    }
    
    if all(v == 0 for v in scores.values()):
        all_fail.append((i, q))
    elif sum(1 for v in scores.values() if v > 0) == 1:
        winner = [k for k, v in scores.items() if v > 0][0]
        only_one_wins.append((i, q, winner))

if all_fail:
    print(f"All methods failed on {len(all_fail)} queries:")
    for i, q in all_fail:
        print(f"  Q{i+1}: {q['query']}")
        print(f"        Relevant: {q['relevant']}")
else:
    print("No queries where all methods failed.")

print()

if only_one_wins:
    print(f"Only one method succeeded on {len(only_one_wins)} queries:")
    for i, q, winner in only_one_wins:
        print(f"  Q{i+1}: {q['query']} --> {winner} only")
else:
    print("No queries where only one method succeeded.")

# Overall winner tally
print("\nPer-query winner tally (by Recall):")
tally = {"BM25": 0, "Dense": 0, "Hybrid": 0, "Tie": 0}
for i in range(len(queries)):
    r = {name: results[name][i]["recall"] for name in ["BM25", "Dense", "Hybrid"]}
    best = max(r.values())
    winners = [k for k, v in r.items() if v == best]
    if len(winners) == 3:
        tally["Tie"] += 1
    else:
        for w in winners:
            tally[w] += 1
for name, count in tally.items():
    print(f"  {name}: {count}")


## Limitations

1. **Small corpus (20 docs).** Real-world search operates on thousands-to-millions.
   Relative rankings may shift at scale.
2. **Single embedding model.** Larger models (e.g., BGE-M3, Qwen3-Embedding)
   may improve dense retrieval significantly.
3. **No reranking.** In production, a cross-encoder reranker on top of hybrid
   retrieval typically adds 5-15% recall.
4. **English-only corpus.** Multilingual content favours dense retrieval even more.
5. **No query expansion.** Techniques like HyDE or query rewriting help BM25.
6. **Binary relevance.** Real systems use graded relevance (highly relevant, somewhat, not).


## Common Mistakes

| Mistake | Why it's wrong | Fix |
|---------|---------------|-----|
| Using only BM25 | Misses semantic/paraphrase queries | Add dense retrieval |
| Using only dense | Misses exact keyword matches | Add BM25 |
| Not normalising embeddings | Cosine similarity requires unit vectors | L2-normalise |
| Evaluating on in-domain only | Overestimates real-world performance | Use varied query types |
| Ignoring latency | Dense retrieval can be 10x slower | Benchmark latency too |
| Tuning RRF k without eval | Affects fusion balance unpredictably | Always evaluate |


## Mini Challenge

1. **Add a cross-encoder reranker** on top of hybrid retrieval.
   Use `cross-encoder/ms-marco-MiniLM-L-6-v2` from sentence-transformers.
   Does it improve MRR?

2. **Try weighted fusion** instead of RRF:
   `score = alpha * bm25_score_norm + (1-alpha) * dense_score_norm`.
   What alpha works best?

3. **Scale up.** Add 50+ documents and see how BM25 vs dense performance changes.

4. **Add NDCG@k.** Implement Normalised Discounted Cumulative Gain
   for a position-sensitive metric.

5. **Test a larger embedding model.** Replace all-MiniLM-L6-v2 with
   all-mpnet-base-v2 and compare.


## Production Considerations

| Aspect | BM25 | Dense | Hybrid |
|--------|------|-------|--------|
| **Latency** | Sub-ms | 5-50ms (depends on index) | Sum of both |
| **Infrastructure** | Elasticsearch / Lucene | Vector DB (Pinecone, Qdrant) | Both |
| **Cost** | Low (CPU only) | Higher (GPU for encoding) | Highest |
| **Maintenance** | Index rebuild on update | Re-embed on model change | Both |
| **Cold start** | Works immediately | Needs embedding model | Both |

**Recommendation:** Start with BM25. Add dense retrieval when semantic
queries are common. Use hybrid with RRF when highest quality matters.


## How to Improve This Project

1. **Cross-encoder reranking** -- Rerank top-k hybrid results with a
   cross-encoder for dramatically better ranking quality.
2. **Learned sparse retrieval** -- SPLADE or ColBERT combine benefits
   of sparse and dense without rank fusion.
3. **Query expansion** -- Use HyDE (hypothetical document embedding)
   to improve both BM25 and dense retrieval.
4. **Production vector DB** -- Replace numpy search with Qdrant,
   Pinecone, or Weaviate for ANN search at scale.
5. **Multilingual evaluation** -- Test with non-English queries
   where dense retrieval should dominate more strongly.
6. **End-to-end RAG evaluation** -- Measure not just retrieval quality
   but the quality of generated answers using the retrieved context.


## Key Takeaways

1. **BM25 excels at exact/keyword queries.** When query terms appear
   literally in target documents, BM25 is hard to beat.

2. **Dense retrieval excels at semantic queries.** It understands meaning
   even when vocabulary differs completely.

3. **Hybrid (RRF) is the most robust.** It's rarely the worst performer
   and often the best. The combination covers both failure modes.

4. **RRF is simple and effective.** No training, no score normalisation,
   just rank-based fusion with one hyperparameter.

5. **BM25 is 10-100x faster** than dense retrieval. For latency-sensitive
   applications, it may still be the right choice.

6. **Always evaluate on multiple query types.** A single aggregate score
   hides critical failure modes.

7. **Start simple, add complexity.** BM25 first, then dense, then hybrid,
   then reranking. Each layer adds cost and complexity.
